# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

sample = pd.DataFrame(list(islice(daily, 5000)))

sample.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 1. Build the feature vector

# Feature Vector

The feature vector contains only historical information that would be available before making a refresh decision.

The selected features are:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- sessions_organic

Missing values are filled using zero because missing values typically indicate unavailable observations rather than future information.

No future labels or product-generated flags are included.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell AB# Select only historical features

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "sessions_organic"
]

features = sample[feature_columns].copy()

# Fill missing values
features = features.fillna(0)

print("Feature Vector Shape:", features.shape)

features.head()

Feature Vector Shape: (5000, 6)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,sessions_organic
0,30,0,3.833333,0,0,0
1,5,0,71.600000,0,0,0
2,1,0,34.000000,0,0,0
3,6,0,23.333333,0,0,0
4,5,0,17.800000,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

## Feature Notes

### gsc_impressions
Meaning:
Number of search impressions from Google Search Console.

Missing:
Filled with 0.

Available when?
Yes. Historical impressions are already known before making a decision.

---

### gsc_clicks
Meaning:
Historical clicks from Google Search Console.

Missing:
Filled with 0.

Available when?
Yes.

---

### gsc_avg_position
Meaning:
Average ranking position in Google Search.

Missing:
Filled with 0.

Available when?
Yes.

---

### ga4_pageviews
Meaning:
Historical page views from Google Analytics 4.

Missing:
Filled with 0.

Available when?
Yes.

---

### ga4_sessions
Meaning:
Historical GA4 sessions.

Missing:
Filled with 0.

Available when?
Yes.

---

### sessions_organic
Meaning:
Organic search sessions.

Missing:
Filled with 0.

Available when?
Yes.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features.info()

features.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gsc_impressions   5000 non-null   int64  
 1   gsc_clicks        5000 non-null   int64  
 2   gsc_avg_position  5000 non-null   float64
 3   ga4_pageviews     5000 non-null   int64  
 4   ga4_sessions      5000 non-null   int64  
 5   sessions_organic  5000 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 234.5 KB


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,sessions_organic
count,5000.000000,5000.000000,5000.000000,5000.0,5000.0,5000.0
mean,11.489400,0.105600,25.718199,0.0,0.0,0.0
std,18.728258,0.421526,23.504413,0.0,0.0,0.0
min,1.000000,0.000000,0.000000,0.0,0.0,0.0
25%,2.000000,0.000000,7.833333,0.0,0.0,0.0
50%,6.000000,0.000000,16.445906,0.0,0.0,0.0
75%,14.000000,0.000000,37.200000,0.0,0.0,0.0
max,424.000000,8.000000,127.000000,0.0,0.0,0.0


## Leakage Hunt

The feature set was reviewed for possible leakage.

The following checks were performed:

- No future performance windows were used.
- No label-derived columns were used.
- No product-generated recommendation flags were included.
- Only historical observations available before the decision point were retained.

These checks reduce the risk of information leakage and help ensure a fair evaluation.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a deliberately bad feature

sample["leak_feature"] = (
    sample["gsc_clicks"]
    >
    sample["gsc_clicks"].median()
).astype(int)

print("Leakage feature created.")

sample[
    [
        "gsc_clicks",
        "leak_feature"
    ]
].head()

# Remove it immediately

sample.drop(
    columns=["leak_feature"],
    inplace=True
)

print("Leakage feature removed.")

Leakage feature created.
Leakage feature removed.


## 4. What I excluded and why

## Excluded Fields

### Future outcomes
Excluded because they would not be available at prediction time.

### Product recommendation flags
Excluded because they already contain decision logic and would introduce leakage.

### Final outcome windows
Excluded because they represent future information.

### Client identifiers
Excluded because they do not represent meaningful predictive features and may encourage memorization rather than generalization.

### Content identifiers
Excluded for the same reason as client identifiers.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("Excluded fields:")

for field in excluded_fields:
    print("-", field)

Excluded fields:
- client_hash_id
- content_hash_id
- report_date


## Self-check

✅ Every section above contains both markdown explanations and supporting code.

✅ The notebook runs from top to bottom without errors.

✅ No client names, URLs, or private search queries are included.

✅ Claims use careful wording:
observed, measured, directional, decision-support.

✅ Notebook committed under:

work/notebooks/w05_feature_vector.ipynb